In [8]:
from playwright.async_api import async_playwright
import asyncio

url = 'https://www.gowork.pl/opinie_czytaj,949234'
print("🕵️ Odpalam wirtualną przeglądarkę (TYM RAZEM WIDOCZNĄ!)...")

playwright = await async_playwright().start()

# 1. ZMIANA: Wracamy do widocznej przeglądarki. Cloudflare rzadziej blokuje widoczne okna.
browser = await playwright.chromium.launch(headless=False)
page = await browser.new_page()

print(f"🌐 Wchodzę na stronę: {url}")
await page.goto(url)

print("⏳ Symuluję ludzkie zachowanie (scrollowanie), żeby wymusić załadowanie opinii...")

try:
    # 2. ZMIANA: Symulujemy przewinięcie strony (scroll) o 2000 pikseli w dół
    await page.mouse.wheel(0, 2000) 
    
    # Dajemy stronie 3 sekundy na zorientowanie się, że zjechaliśmy w dół i załadowanie danych
    await asyncio.sleep(3) 

    nasz_celownik = 'span.MuiTypography-body1.whitespace-pre-wrap'
    
    print("🎯 Namierzam opinie...")
    await page.wait_for_selector(nasz_celownik, timeout=15000)
    
    # 3. Zgarniamy wszystko!
    lista_opinii = await page.locator(nasz_celownik).all_inner_texts()
    
    print("\n" + "=" * 50)
    print(f"🔥 POTĘŻNY SUKCES! Zassałem {len(lista_opinii)} opinii!")
    print("=" * 50 + "\n")
    
    print("Oto próbka danych (pierwsze 3 opinie):\n")
    for i, opinia in enumerate(lista_opinii[:3], 1):
        print(f"--- OPINIA NR {i} ---")
        print(opinia.strip())
        print("-" * 30 + "\n")
        
except Exception as e:
    print("\n❌ Niestety, wystąpił błąd. Zobacz w otwartym oknie przeglądarki, co zablokowało skrypt:")
    print(e)
    
finally:
    await browser.close()
    await playwright.stop()
    print("🧹 Przeglądarka zamknięta, misja zakończona.")

🕵️ Odpalam wirtualną przeglądarkę (TYM RAZEM WIDOCZNĄ!)...
🌐 Wchodzę na stronę: https://www.gowork.pl/opinie_czytaj,949234
⏳ Symuluję ludzkie zachowanie (scrollowanie), żeby wymusić załadowanie opinii...
🎯 Namierzam opinie...

🔥 POTĘŻNY SUKCES! Zassałem 57 opinii!

Oto próbka danych (pierwsze 3 opinie):

--- OPINIA NR 1 ---
Jaki jest model pracy dla programistów? W ofercie niestety brak konkretnych informacji. Zdalnie/hybryda/z biura ewentualnie w jakich proporcjach?
------------------------------

--- OPINIA NR 2 ---
Z tego co widzę to na pracuj wszędzie jest wpisana hybryda, ale to tyle 😅 Może ktoś da znać ile dni tygodniowo pracujecie z biura a ile na HO? I w sumie czy jest opcja dogadać się na pracę w większości zdalną? 😀
------------------------------

--- OPINIA NR 3 ---
W UE i w Polsce za nieuczciwą moderację (np. arbitralne kasowanie opinii, komentarzy i kont) platformom takim jak Google/YouTube grożą wysokie kary finansowe, nakazy zmian procedur oraz obowiązki zapewnienia skut

In [12]:
import pandas as pd
import os

# 1. Definiujemy nazwę folderu
folder_name = 'opinie'

# 2. Sprawdzamy, czy folder istnieje. Jeśli nie – tworzymy go!
# 'exist_ok=True' sprawi, że Python nie zgłosi błędu, jeśli folder już tam jest.
os.makedirs(folder_name, exist_ok=True)

# 3. Tworzymy DataFrame (zakładając, że masz listę_opinii z poprzedniego kroku)
df = pd.DataFrame(lista_opinii, columns=['Tresc_Opinii'])
df = df.replace('', pd.NA).dropna()

# 4. Zapisujemy do podfolderu
# Używamy '/' aby oddzielić folder od nazwy pliku
file_path = os.path.join(folder_name, 'opinie_gowork.csv')

df.to_csv(file_path, index=False, encoding='utf-8-sig')

print(f"💾 Sukces! Plik został zapisany w: {file_path}")

💾 Sukces! Plik został zapisany w: opinie/opinie_gowork.csv
